<a href="https://colab.research.google.com/github/vishakha19-hue/vishakha/blob/main/saleprediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
"""
CodeAlpha Data Science Internship
TASK 4: Sales Prediction using Python
-----------------------------------------
Goal: Predict product sales based on advertising spend across
TV, Radio, and Newspaper, plus target segment and platform.
"""

import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

os.makedirs('./outputs', exist_ok=True)

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score, mean_absolute_error, mean_squared_error

np.random.seed(7)

# ---------------------------------------------------------
# STEP 1: Create Dataset
# ---------------------------------------------------------
n = 500
segments = ['Youth', 'Adults', 'Seniors']
platforms = ['TV', 'Social Media', 'Print', 'Mixed']

df = pd.DataFrame({
    'TV_spend': np.round(np.random.uniform(0, 300, n), 1),
    'Radio_spend': np.round(np.random.uniform(0, 50, n), 1),
    'Newspaper_spend': np.round(np.random.uniform(0, 100, n), 1),
    'target_segment': np.random.choice(segments, n),
    'platform': np.random.choice(platforms, n),
})

segment_factor = {'Youth': 1.2, 'Adults': 1.0, 'Seniors': 0.8}
platform_factor = {'TV': 1.1, 'Social Media': 1.3, 'Print': 0.7, 'Mixed': 1.0}

df['seg_f'] = df['target_segment'].map(segment_factor)
df['plat_f'] = df['platform'].map(platform_factor)

# Sales driven mostly by TV & Radio spend (diminishing returns), some noise
sales = (
    4
    + 0.045 * df['TV_spend']
    + 0.19 * df['Radio_spend']
    + 0.01 * df['Newspaper_spend']
    + 0.002 * df['TV_spend'] * df['Radio_spend'] / 10
) * df['seg_f'] * df['plat_f']

noise = np.random.normal(0, 1.2, n)
df['Sales'] = (sales + noise).clip(lower=0.5).round(2)
df.drop(columns=['seg_f', 'plat_f'], inplace=True)

print("First 5 rows of dataset:")
print(df.head())
print("\nDataset shape:", df.shape)
print("\nSales statistics:\n", df['Sales'].describe())

# ---------------------------------------------------------
# STEP 2: EDA
# ---------------------------------------------------------
plt.figure(figsize=(8, 6))
sns.heatmap(df[['TV_spend', 'Radio_spend', 'Newspaper_spend', 'Sales']].corr(),
            annot=True, cmap='coolwarm')
plt.title("Correlation: Ad Spend vs Sales")
plt.savefig('./outputs/task4_correlation.png', dpi=120, bbox_inches='tight')
plt.close()

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
for ax, col in zip(axes, ['TV_spend', 'Radio_spend', 'Newspaper_spend']):
    sns.scatterplot(data=df, x=col, y='Sales', ax=ax, alpha=0.5)
    ax.set_title(f"{col} vs Sales")
plt.tight_layout()
plt.savefig('./outputs/task4_spend_vs_sales.png', dpi=120, bbox_inches='tight')
plt.close()

# ---------------------------------------------------------
# STEP 3: Preprocessing
# ---------------------------------------------------------
numeric_features = ['TV_spend', 'Radio_spend', 'Newspaper_spend']
categorical_features = ['target_segment', 'platform']

X = df.drop(columns=['Sales'])
y = df['Sales']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

preprocessor = ColumnTransformer(transformers=[
    ('num', StandardScaler(), numeric_features),
    ('cat', OneHotEncoder(handle_unknown='ignore'), categorical_features)
])

# ---------------------------------------------------------
# STEP 4: Train Models
# ---------------------------------------------------------
lin_pipeline = Pipeline([('prep', preprocessor), ('model', LinearRegression())])
lin_pipeline.fit(X_train, y_train)
lin_pred = lin_pipeline.predict(X_test)

rf_pipeline = Pipeline([('prep', preprocessor), ('model', RandomForestRegressor(n_estimators=200, random_state=42))])
rf_pipeline.fit(X_train, y_train)
rf_pred = rf_pipeline.predict(X_test)

# ---------------------------------------------------------
# STEP 5: Evaluate
# ---------------------------------------------------------
def evaluate(name, y_true, y_pred):
    r2 = r2_score(y_true, y_pred)
    mae = mean_absolute_error(y_true, y_pred)
    rmse = np.sqrt(mean_squared_error(y_true, y_pred))
    print(f"\n===== {name} =====")
    print(f"R2 Score : {r2:.4f}")
    print(f"MAE      : {mae:.3f}")
    print(f"RMSE     : {rmse:.3f}")
    return r2

r2_lin = evaluate("Linear Regression", y_test, lin_pred)
r2_rf = evaluate("Random Forest Regressor", y_test, rf_pred)

best_pred = rf_pred if r2_rf >= r2_lin else lin_pred
best_name = "Random Forest" if r2_rf >= r2_lin else "Linear Regression"

plt.figure(figsize=(7, 7))
plt.scatter(y_test, best_pred, alpha=0.5, color='crimson')
plt.plot([y_test.min(), y_test.max()], [y_test.min(), y_test.max()], 'k--', lw=2)
plt.xlabel("Actual Sales")
plt.ylabel("Predicted Sales")
plt.title(f"Actual vs Predicted Sales ({best_name})")
plt.savefig('./outputs/task4_actual_vs_predicted.png', dpi=120, bbox_inches='tight')
plt.close()

print(f"\nBest model: {best_name}")
print("All plots saved to outputs folder.")
print("TASK 4 COMPLETE ✅")


First 5 rows of dataset:
   TV_spend  Radio_spend  Newspaper_spend target_segment      platform  Sales
0      22.9         48.2              8.4        Seniors         Print   7.79
1     234.0         24.3             52.6        Seniors         Print  11.35
2     131.5         23.9             38.2         Adults  Social Media  18.93
3     217.0         45.1             81.8         Adults         Print  17.04
4     293.4          0.8             28.4          Youth  Social Media  29.59

Dataset shape: (500, 6)

Sales statistics:
 count    500.000000
mean      17.327880
std        7.269654
min        3.000000
25%       11.957500
50%       16.735000
75%       21.865000
max       44.980000
Name: Sales, dtype: float64

===== Linear Regression =====
R2 Score : 0.9219
MAE      : 1.525
RMSE     : 2.061

===== Random Forest Regressor =====
R2 Score : 0.8933
MAE      : 1.949
RMSE     : 2.409

Best model: Linear Regression
All plots saved to outputs folder.
TASK 4 COMPLETE ✅
